# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results.

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 

In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

!pip install numpy
!pip install pandas
!pip install seaborn
!pip install matplotlib.pyplot
!pip install matplotlib.patches
!pip install matplotlib.ticker
!pip install scikit-learn
!pip install tqdm

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


ERROR: Could not find a version that satisfies the requirement matplotlib.pyplot (from versions: none)
ERROR: No matching distribution found for matplotlib.pyplot
ERROR: Could not find a version that satisfies the requirement matplotlib.patches (from versions: none)
ERROR: No matching distribution found for matplotlib.patches
ERROR: Could not find a version that satisfies the requirement matplotlib.ticker (from versions: none)
ERROR: No matching distribution found for matplotlib.ticker


### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you'll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here's what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it's called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.

6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()
>   X_train_scaled = scaler.fit_transform(X_train)

**Notes:**

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).

In [2]:
df = pd.read_csv("zillow_cleaned.csv")

target = 'taxvaluedollarcnt'
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 

In [3]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [4]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=42),
}

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train,
                             scoring='neg_mean_absolute_error',
                             cv=cv, n_jobs=-1)
    mae_scores = -scores
    results[name] = (mae_scores.mean(), mae_scores.std())
    print(f"{name}: MAE = {mae_scores.mean():,.2f} ± {mae_scores.std():,.2f}")


Linear Regression: MAE = 170,955.87 ± 1,532.23
Random Forest: MAE = 150,768.52 ± 1,213.23
HistGradientBoosting: MAE = 150,132.68 ± 1,107.20


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

Best overall performance: HistGradientBoosting achieved the lowest MAE ($150,133), narrowly outperforming Random Forest ($150,769). Both ensemble methods substantially outperformed Linear Regression ($170,956), reducing error by roughly $20,000.
Most stable model: HistGradientBoosting also had the lowest standard deviation (±$1,107), indicating the most consistent performance across CV folds. Random Forest was close behind (±$1,213), while Linear Regression showed the most variability (±$1,532).
Overfitting/underfitting: Linear Regression's significantly higher MAE suggests it is underfitting, it cannot capture the nonlinear relationships in housing prices. The two ensemble methods perform similarly to each other, which suggests they are capturing the underlying patterns well without obvious signs of overfitting, as their CV scores remain stable across folds and repeats.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1.

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler`
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  

In [5]:
X_train_eng = X_train.copy()
X_test_eng = X_test.copy()

X_train_eng['log_sqft'] = np.log1p(X_train_eng['calculatedfinishedsquarefeet'].clip(lower=0))
X_test_eng['log_sqft'] = np.log1p(X_test_eng['calculatedfinishedsquarefeet'].clip(lower=0))

X_train_eng['home_age'] = 2017 - X_train_eng['yearbuilt']
X_test_eng['home_age'] = 2017 - X_test_eng['yearbuilt']

X_train_eng['total_rooms'] = X_train_eng['bathroomcnt'] + X_train_eng['bedroomcnt']
X_test_eng['total_rooms'] = X_test_eng['bathroomcnt'] + X_test_eng['bedroomcnt']

scaler_eng = StandardScaler()
X_train_eng_scaled = scaler_eng.fit_transform(X_train_eng)
X_test_eng_scaled = scaler_eng.transform(X_test_eng)

for name, model in models.items():
    scores = cross_val_score(model, X_train_eng_scaled, y_train,
                             scoring='neg_mean_absolute_error',
                             cv=cv, n_jobs=-1)
    mae_scores = -scores
    print(f"{name}: MAE = {mae_scores.mean():,.2f} ± {mae_scores.std():,.2f}")


Linear Regression: MAE = 170,999.37 ± 1,547.86
Random Forest: MAE = 150,824.54 ± 1,228.39
HistGradientBoosting: MAE = 150,164.41 ± 1,142.92


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn't)?



No notable improvement: The engineered features did not meaningfully improve any of the three models. Linear Regression went from $170,956 to $170,999 (slightly worse), Random Forest from $150,769 to $150,825 (slightly worse), and HistGradientBoosting from $150,133 to $150,164 (essentially flat).
Why the features may not have helped: The total_rooms feature is a simple sum of bathroomcnt and bedroomcnt, which the models, especially the tree-based ones, can already learn implicitly through splits on those individual features. Similarly, home_age is a linear transformation of yearbuilt, so it carries no new information for Linear Regression and is redundant for tree models. The log_sqft transform helps linearize the relationship, but the tree-based models don't benefit from this since they handle nonlinearity naturally.
Hypothesis: These engineered features primarily aid linear models by reshaping relationships, but the improvement wasn't enough to overcome Linear Regression's fundamental limitation with this dataset. For ensemble methods, the original features already provided sufficient signal. More impactful features might involve interactions like price-per-square-foot by region or location-based groupings.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  

In [6]:
# Add as many cells as you need
from sklearn.inspection import permutation_importance

f_n = pd.Index(X_train_eng.columns)
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state = 42)
s = SelectKBest(f_regression, k=5).fit(X_train_eng_scaled, y_train)
X_train_lr = s.transform(X_train_eng_scaled)
lr_features = f_n[s.get_support()]

lr_scores = cross_val_score(LinearRegression(), X_train_lr, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
lr_mae = lr_scores * -1
print(f"Linear Regression , MAE = {lr_mae.mean():,.2f} +/-{lr_mae.std():,.2f}")
print(f"Features: {lr_features.tolist()}\n")

rf_base = RandomForestRegressor(random_state=42, n_jobs=-1).fit(X_train_eng_scaled, y_train)
rf_imp = pd.Series(rf_base.feature_importances_,index=f_n)
rf_mask = f_n.isin(rf_imp.head(5).index)
X_train_rf = X_train_eng_scaled[:, rf_mask]
rf_scores = cross_val_score(RandomForestRegressor(random_state=42, n_jobs=-1), X_train_rf, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)

rf_mae = rf_scores * -1
print(f"Linear Regression , MAE = {lr_mae.mean():,.2f} +/-{rf_mae.std():,.2f}")
print(f"Features: {rf_imp.head(5).index.tolist()}\n")

hgb_base = HistGradientBoostingRegressor(random_state=42).fit(X_train_eng_scaled, y_train)
hgb_perm = permutation_importance(hgb_base, X_train_eng_scaled, y_train, random_state=42, n_jobs = -1)
hgb_imp = pd.Series(hgb_perm.importances_mean, index=f_n).sort_values(ascending=False)
hgb_mask = f_n.isin(hgb_imp.head(5).index)
X_train_hgb = X_train_eng_scaled[:, hgb_mask]
hgb_scores = cross_val_score(HistGradientBoostingRegressor(random_state=42), X_train_hgb, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
hgb_mae = hgb_scores * -1
print(f"HistGradientBossting, MAE = {hgb_mae.mean():,.2f}+/-{hgb_mae.std():,.2f}")
print(f"Features: {hgb_imp.head(5).index.tolist()}\n")

Linear Regression , MAE = 183,538.58 +/-1,417.81
Features: ['bathroomcnt', 'calculatedbathnbr', 'calculatedfinishedsquarefeet', 'fullbathcnt', 'log_sqft']

Linear Regression , MAE = 183,538.58 +/-1,537.62
Features: ['bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'calculatedfinishedsquarefeet']

HistGradientBossting, MAE = 151,604.57+/-1,185.31
Features: ['calculatedfinishedsquarefeet', 'latitude', 'longitude', 'yearbuilt', 'regionidzip']



### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?

The performance did not have major improvements after feature selection. The MAE is 183, 528.57 is definately worse than the MAE from part 2 of 170,999. Overall reducing to 5 features has taken out important information negatively impacting all 3 models. Caclulatedfinishedsquarefeet as well as bathroomcnt appeared the most amongst all 3 models. Calculatedbathnbr appeared the second most often, showing a very close correlation to the other features. The selectkbest choosed log_sqft. In the long run, this means that the logarithmic transformation helps showcase the relationships in a linear formation for the square feet discussion. At the same time, it is also important to note that total_rooms and/or home_age were not taken into consideration by any of the models in the long run consideration of important features.

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you'll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above.
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks.
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [7]:
# Add as many cells as you need

lr_s = cross_val_score(LinearRegression(), X_train_lr, y_train, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
lr_mae = lr_s * -1
print(f"LinearRegression, MAE = {lr_mae.mean():,.2f} +/- {(lr_s * -1).std():,.2f}")

rf_s = GridSearchCV(RandomForestRegressor(random_state=42), {'n_estimators': [100,200], 'max_depth': [None, 10, 20]}, scoring ='neg_mean_absolute_error', cv=cv, n_jobs=-1)
rf_s.fit(X_train_rf, y_train)
print(f"Random Forest, MAE={rf_s.best_score_*-1:,.2f}, Best- {rf_s.best_params_}")

hgb_s = GridSearchCV(HistGradientBoostingRegressor(random_state=42), {'learning_rate': [0.05, 0.1, 0.2], 'max_iter': [100, 200]}, scoring ='neg_mean_absolute_error', cv=cv, n_jobs=-1)
hgb_s.fit(X_train_hgb, y_train)
print(f"HistGradientBoosting, MAE={hgb_s.best_score_*-1:,.2f}, Best- {hgb_s.best_params_}")

LinearRegression, MAE = 183,538.58 +/- 1,417.81
Random Forest, MAE=174,935.85, Best- {'max_depth': 10, 'n_estimators': 200}
HistGradientBoosting, MAE=151,294.25, Best- {'learning_rate': 0.1, 'max_iter': 200}


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?

For the tuning stategy for each model there were different approaches. The linear regression did not have any hyperparameters to tune, but selectkbest here selected the upmost 5 features. For the random forest, gridsearchcv used the estimators and depth, for a overall of 10 and 200 in that order. This resulted in a MAE of 174,935.85. Here the data prevented the overfitting as a result. For Histgradientboosting, again, gridsearchcv looked over the max iter and learning rate, resulting in 200 and 0.1 in that order. This gave a result of 151,294.25 as a result which is close to the baseline value from part 2. 
One thing to note here is that log sqft was helpful for the linear regression to showcase the relationships. This means that linear regression was slightly more sensitive to the feature selections as compared to the other models.

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You'll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants.

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.

    - Test score on held-out test set.



In [9]:
from sklearn.metrics import mean_absolute_error

X_test_hgb = X_test_eng_scaled[:, hgb_mask]

final_model = HistGradientBoostingRegressor(
    learning_rate=0.1,
    max_iter=200,
    random_state=42
)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
cv_scores = cross_val_score(final_model, X_train_hgb, y_train,
                             scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
cv_mae = cv_scores * -1
print(f"CV MAE: {cv_mae.mean():,.2f} +/- {cv_mae.std():,.2f}")

final_model.fit(X_train_hgb, y_train)
y_pred = final_model.predict(X_test_hgb)
test_mae = mean_absolute_error(y_test, y_pred)
print(f"Test MAE: {test_mae:,.2f}")

CV MAE: 151,294.25 +/- 1,145.64
Test MAE: 150,590.09


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

1.
The final modwel we chose was the HistGradientBoostingRegressor with learning rate 0.1 and max_iter 200. The metrics that led to this decision was that it had the lowest MAE out of all the other models with an MAE of 151,294. The tradeoff for this model was interpretability but the performance boost was really good for the tradeoff.

2.
One specific preprocessing deciosn that was really important was making the log sqft feature because home prices will not scale linearly with size. It helped scale the prices better linearly so that linear regression would benefit more which was why we did it. This step really only benefitted linear regression marginally and did not really impact any of the other models. We still kept it because it didn't do that much and the MAE only slightly went down.

3.
An important insight about the dataset was that geographical features like longitude and latitude were really critical for HistGradiantBoosting but weren't important for Linear regression. That means that these features were super non linear and that these trends were unable to be captured by simple models. What I would explore next would be models that could try to capture these trends better as I think location should contribute a lot to a houses price.